# Walker2d-v4 Training | Kaggle
Methodology v12.2 | CPU-only torch | skip-if-exists checkpointing

In [ ]:
# Install CPU-only torch and dependencies
!pip install torch --index-url https://download.pytorch.org/whl/cpu -q
!pip install stable-baselines3 gymnasium[mujoco] -q


In [ ]:
%%writefile train_walker2d.py
"""
train_walker2d.py — Walker2d-v4 training script
Methodology: v12.2 | Platform: Kaggle (CPU)
Conditions: M1, M2, M3 | Seeds: 5 | Budget: 1,000,000 steps/seed/condition
"""

import os
import sys
import time
import torch
import numpy as np
import gymnasium as gym
from gymnasium.wrappers import TimeLimit
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv

# ── Reproducibility ────────────────────────────────────────────────────────────
torch.set_num_threads(os.cpu_count())

# ── Config ─────────────────────────────────────────────────────────────────────
OUTPUT_DIR = "./results/walker2d"
TOTAL_STEPS = 1_000_000
N_ENVS = 3
SEEDS = [0, 1, 2, 3, 4]

CONDITIONS = {
    "M1": {"time_limit": 1000, "reset_noise_scale": 0.005},
    "M2": {"time_limit": 250,  "reset_noise_scale": 0.005},
    "M3": {"time_limit": 1000, "reset_noise_scale": 0.02},
}

PPO_KWARGS = dict(
    learning_rate   = 3e-4,
    n_steps         = 2048,
    batch_size      = 128,
    n_epochs        = 10,
    gamma           = 0.99,
    gae_lambda      = 0.95,
    clip_range      = 0.2,
    clip_range_vf   = None,
    ent_coef        = 0.0,
    vf_coef         = 0.5,
    max_grad_norm   = 0.5,
    normalize_advantage = True,
    use_sde         = False,
    target_kl       = None,
    policy_kwargs       = dict(
        net_arch       = dict(pi=[256, 256], vf=[256, 256]),
        activation_fn  = torch.nn.ReLU,
        optimizer_class= torch.optim.Adam,
    ),
    verbose         = 1,
    device          = "cpu",
)

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ── Environment factory ─────────────────────────────────────────────────────────
def make_env_fn(time_limit, reset_noise_scale, seed):
    def _init():
        env = gym.make(
            "Walker2d-v4",
            reset_noise_scale=reset_noise_scale,
        )
        env = TimeLimit(env.unwrapped, max_episode_steps=time_limit)
        env.reset(seed=seed)
        return env
    return _init


# ── Training loop ──────────────────────────────────────────────────────────────
print("=" * 60)
print("WALKER2D-v4 TRAINING")
print(f"Conditions: {list(CONDITIONS.keys())} | Seeds: {SEEDS}")
print(f"Steps/run: {TOTAL_STEPS:,} | n_envs: {N_ENVS} | Device: cpu")
print("=" * 60)

total_start = time.time()

for cond_name, cond_cfg in CONDITIONS.items():
    print(f"\n── Condition {cond_name} | TimeLimit={cond_cfg['time_limit']} | "
          f"reset_noise={cond_cfg['reset_noise_scale']} ──")
    cond_start = time.time()

    for seed in SEEDS:
        save_path = os.path.join(OUTPUT_DIR, f"{cond_name}_seed{seed}.zip")

        if os.path.exists(save_path):
            print(f"  [skip] {cond_name} seed={seed} — already trained")
            continue

        run_start = time.time()
        print(f"  [train] {cond_name} seed={seed} ...", flush=True)

        vec_env = DummyVecEnv([
            make_env_fn(cond_cfg["time_limit"], cond_cfg["reset_noise_scale"], seed + i * 1000)
            for i in range(N_ENVS)
        ])

        model = PPO("MlpPolicy", vec_env, seed=seed, **PPO_KWARGS)
        model.learn(total_timesteps=TOTAL_STEPS)
        model.save(save_path)
        vec_env.close()

        elapsed = time.time() - run_start
        print(f"  [done]  {cond_name} seed={seed} → {save_path} ({elapsed/60:.1f} min)")

    cond_elapsed = time.time() - cond_start
    print(f"  Condition {cond_name} complete in {cond_elapsed/60:.1f} min")

total_elapsed = time.time() - total_start
print(f"\n{'=' * 60}")
print(f"All training complete in {total_elapsed/60:.1f} min")
print(f"{'=' * 60}")

# ── Inventory ──────────────────────────────────────────────────────────────────
print("\nOutput inventory:")
for cond_name in CONDITIONS:
    for seed in SEEDS:
        path = os.path.join(OUTPUT_DIR, f"{cond_name}_seed{seed}.zip")
        status = "✓" if os.path.exists(path) else "✗ MISSING"
        print(f"  {status}  {cond_name}_seed{seed}.zip")


In [ ]:
!python train_walker2d.py
